# 03. Mô hình Image-Only & Vision-Centric cho ViMMSD

Notebook này thực hiện huấn luyện và **đánh giá toàn diện trên Precision, Recall, F1-Score** cho 3 phương pháp:
1. **Task 5 & Task 8**: Mô hình Image-Only dùng **CLIP** (`openai/clip-vit-base-patch32`) và đánh giá (P, R, F1).
2. **Task 6 & Task 9**: Mô hình Image-Only dùng **ViT thuần** (`google/vit-base-patch16-224`) và đánh giá (P, R, F1).
3. **Task 7 & Task 10**: Mô hình Vision-Centric dùng **VLM với Metadata** (Vintern-1B / Qwen2.5-VL + OCR & mô tả ngữ cảnh) và đánh giá (P, R, F1).

In [ ]:
# === Cell bootstrap (giống nhau ở mọi notebook, xem CODEBASE.md mục 6) ===
import os, subprocess, sys
from pathlib import Path

REPO = "github.com/yunaLee21/Multimodal_Sarcasm_Detection.git"
REF = "main"               # branch của mình, hoặc 1 commit hash để tái lặp kết quả
CODE_DIR = "/tmp/IE403"    # clone ra ngoài /kaggle/working để token không bị lưu vào output


def get_github_token():
    """Token cho repo private; repo public thì không cần tạo secret."""
    try:
        if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("GITHUB_TOKEN")
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None


ON_CLOUD = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or "COLAB_RELEASE_TAG" in os.environ
if ON_CLOUD:
    if "COLAB_RELEASE_TAG" in os.environ and not os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):  # ảnh Kaggle dựng từ ảnh Colab nên cũng có biến này
        from google.colab import drive
        drive.mount("/content/drive")
    token = get_github_token()
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
    subprocess.run(["rm", "-rf", CODE_DIR])
    subprocess.run(["git", "clone", "-q", url, CODE_DIR], check=True)
    subprocess.run(["git", "-C", CODE_DIR, "checkout", "-q", REF], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE_DIR}/requirements-kaggle.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE_DIR], check=True)
    if CODE_DIR + "/src" not in sys.path:
        sys.path.insert(0, CODE_DIR + "/src")  # kernel đang chạy chưa thấy package vừa cài -e
    os.chdir(CODE_DIR)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)  # local: chạy từ gốc repo (đã `pip install -e .` một lần)

print("cwd:", os.getcwd())
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code commit:", commit or "unknown")

In [ ]:
import json
import logging
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import HTML, Image as IPImage, display

from vimmsd.utils.config import load_config
from vimmsd.training.metrics import format_metrics

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
logging.getLogger("httpx").setLevel(logging.WARNING)

---
## 1. Task 5 & Task 8: Huấn luyện & Đánh giá CLIP Image-only (openai/clip-vit-base-patch32)

* **Task 5 (Code/Train)**: Fine-tune CLIP vision tower với Classification Head.
* **Task 8 (Evaluation)**: Đánh giá chi tiết trên các chỉ số **Macro-Precision**, **Macro-Recall**, **Macro-F1** và **Per-class (P, R, F1)** trên tập test có nhãn.

In [ ]:
# Task 5: Huấn luyện CLIP Image-only
!python scripts/train.py --config configs/image_only.yaml --note "Task 5: CLIP Image-only"

In [ ]:
# Task 8: Đánh giá Precision, Recall, F1-score của CLIP
cfg_clip = load_config("configs/image_only.yaml")
out_clip = Path(cfg_clip.paths.output_dir) / cfg_clip.name
!python scripts/evaluate.py --config configs/image_only.yaml --split test

if (out_clip / "test_metrics.json").exists():
    metrics_clip = json.loads((out_clip / "test_metrics.json").read_text())
    print("=== BÁO CÁO ĐÁNH GIÁ TASK 8: CLIP IMAGE-ONLY ===")
    print(format_metrics(metrics_clip))
    if (out_clip / "confusion_matrix_test.png").exists():
        display(IPImage(str(out_clip / "confusion_matrix_test.png"), width=450))
else:
    print(f"Chưa có kết quả tại {out_clip}. Hãy hoàn thành huấn luyện trước.")

---
## 2. Task 6 & Task 9: Huấn luyện & Đánh giá ViT Image-only (google/vit-base-patch16-224)

* **Task 6 (Code/Train)**: Fine-tune Vision Transformer thuần túy (ImageNet-pretrained) + Classification Head.
* **Task 9 (Evaluation)**: Đánh giá chi tiết trên **Precision**, **Recall**, **F1-score** để so sánh xem khi không có semantic alignment với văn bản, mô hình bắt các lớp mỉa mai ra sao.

In [ ]:
# Task 6: Huấn luyện ViT Image-only
!python scripts/train.py --config configs/image_only_vit.yaml --note "Task 6: ViT Image-only"

In [ ]:
# Task 9: Đánh giá Precision, Recall, F1-score của ViT
cfg_vit = load_config("configs/image_only_vit.yaml")
out_vit = Path(cfg_vit.paths.output_dir) / cfg_vit.name
!python scripts/evaluate.py --config configs/image_only_vit.yaml --split test

if (out_vit / "test_metrics.json").exists():
    metrics_vit = json.loads((out_vit / "test_metrics.json").read_text())
    print("=== BÁO CÁO ĐÁNH GIÁ TASK 9: ViT IMAGE-ONLY ===")
    print(format_metrics(metrics_vit))
    if (out_vit / "confusion_matrix_test.png").exists():
        display(IPImage(str(out_vit / "confusion_matrix_test.png"), width=450))
else:
    print(f"Chưa có kết quả tại {out_vit}. Hãy hoàn thành huấn luyện trước.")

---
## 3. Task 7 & Task 10: Khai thác & Đánh giá VLM với Metadata (Vision-Language Model)

* **Task 7**: Sử dụng VLM (Vintern-1B / Qwen2.5-VL) kết hợp metadata (chữ trong ảnh trích từ OCR và mô tả ngữ cảnh hình ảnh).
* **Task 10 (Evaluation)**: Đo lường độ chính xác **Precision**, **Recall**, **F1-score** của VLM trong việc phân biệt 4 sắc thái mỉa mai.

In [ ]:
# Task 10: Đánh giá VLM Zero-shot / Few-shot trên tập test (yêu cầu GPU T4 / P100 cho model 1B-3B)
import torch
from vimmsd.data.dataset import load_all_records
from vimmsd.analysis.llm_compare import HFVisionLanguageClassifier, evaluate_llm, select_few_shot_examples

cfg = load_config("configs/base.yaml")
records = load_all_records(cfg)
test_records = records["test"]
labels = list(cfg.data.labels)
out_vlm = Path(cfg.paths.output_dir) / "vlm_image_centric"
out_vlm.mkdir(parents=True, exist_ok=True)

print(f"Số mẫu đánh giá test: {len(test_records)}")
# Chạy suy luận nếu có GPU, lưu kết quả ra test_metrics.json
# vlm = HFVisionLanguageClassifier("5CD-AI/Vintern-1B-v3_5", labels=labels)
# vlm_metrics, _ = evaluate_llm(vlm, test_records, labels, out_path=out_vlm / "vlm_preds.jsonl")
# (out_vlm / "test_metrics.json").write_text(json.dumps(vlm_metrics, indent=2))

---
## 4. Bảng tổng hợp & So sánh toàn diện (Tasks 8, 9, 10 Comparison)

Đối sánh chi tiết **Macro Metrics** và **Per-Class Precision, Recall, F1** của cả 3 mô hình.

In [ ]:
# Tổng hợp Bảng Macro Metrics & Bảng Chi tiết từng lớp
models = [
    ("CLIP Image-only (Task 8)", out_clip),
    ("ViT Image-only (Task 9)", out_vit),
    ("VLM + Metadata (Task 10)", out_vlm),
]

macro_rows, per_class_rows = [], []
for name, p in models:
    metric_file = p / "test_metrics.json"
    if metric_file.exists():
        m = json.loads(metric_file.read_text())
        macro_rows.append({
            "Mô hình": name,
            "Macro-F1": m.get("macro_f1", 0),
            "Macro-Precision": m.get("macro_precision", 0),
            "Macro-Recall": m.get("macro_recall", 0),
            "Accuracy": m.get("accuracy", 0),
            "Weighted-F1": m.get("weighted_f1", 0),
        })
        for cls_name, cls_m in m.get("per_class", {}).items():
            per_class_rows.append({
                "Mô hình": name,
                "Lớp nhãn": cls_name,
                "Precision": cls_m.get("precision", 0),
                "Recall": cls_m.get("recall", 0),
                "F1-Score": cls_m.get("f1", 0),
                "Số mẫu (Support)": cls_m.get("support", 0),
            })

if macro_rows:
    print("=== BẢNG 1: TỔNG HỢP MACRO METRICS (TASKS 8, 9, 10) ===")
    df_macro = pd.DataFrame(macro_rows).set_index("Mô hình").round(4)
    display(df_macro)
    
    if per_class_rows:
        print("\n=== BẢNG 2: CHI TIẾT PRECISION / RECALL / F1 THEO TỪNG LỚP ===")
        df_class = pd.DataFrame(per_class_rows).pivot(index="Mô hình", columns="Lớp nhãn", values=["Precision", "Recall", "F1-Score"]).round(4)
        display(df_class)
else:
    print("Chưa có file test_metrics.json nào được tạo. Hãy chạy các cell đánh giá ở trên.")

---
## 5. Phân tích lỗi (Error Analysis & Confusion Matrix Breakdown)

Trích xuất các ca dự đoán sai để phân tích nguyên nhân tại sao mô hình nhìn ảnh nhầm lẫn giữa các lớp.

In [ ]:
for name, p in [("CLIP", out_clip), ("ViT", out_vit)]:
    pred_file = p / "test_predictions.csv"
    if pred_file.exists():
        pred_df = pd.read_csv(pred_file)
        wrong = pred_df[pred_df.label != pred_df.pred]
        print(f"\n[Phân tích lỗi {name}] Tỉ lệ sai: {len(wrong)}/{len(pred_df)} ({len(wrong)/len(pred_df)*100:.1f}%)")
        display(wrong.groupby(["label", "pred"]).size().unstack(fill_value=0))
        print(f"10 mẫu sai ngẫu nhiên của {name}:")
        display(wrong.sample(min(10, len(wrong)), random_state=42)[["id", "label", "pred", "caption"]])